# PydanticAI

In [2]:
!uv add "pydantic-ai"

Resolved 248 packages in 1.48s                                       
⠙ Preparing packages... (0/101)                                                 ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/101)-----------------     0 B/139.02 KiB          
⠙ Preparing packages... (0/101)-----------------     0 B/139.02 KiB          
pyasn1               ------------------------------     0 B/82.03 KiB
⠙ Preparing packages... (0/101)-----------------     0 B/139.02 KiB          
pyasn1               ------------------------------     0 B/82.03 KiB
⠙ Preparing packages... (0/101)-----------------     0 B/139.02 KiB          
pyasn1               ------------------------------     0 B/82.03 KiB
⠙ Preparing packages... (0/101)-----------------     0 B/139.02 KiB          
starlette            ------------------------------     0 B/70.95 KiB
pyasn1               ------------------------------     0 B/82.03 KiB
⠙ Preparing packages... (0/10

In [9]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

In [10]:
file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

In [11]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords)
)

In [12]:
from typing import Any, Dict, List

class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """
    def __init__(self, index: Any, highlighter: Any, file_index: Dict[str, str]) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"


In [14]:
search_tools = SearchTools(index=index, highlighter=highlighter, file_index=file_index) 

tools = [search_tools.search, search_tools.get_file]


In [15]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [16]:
from pydantic_ai import Agent

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)


In [17]:
query = 'how do I use evidently to monitor my machine learning models?'

In [18]:
result = await search_agent.run(query)

In [19]:
print(result.output)

From the retrieved documents, the "Overview" document explains how monitoring works within the system. It details methods such as batch monitoring jobs and tracing with scheduled evaluations, which are vital for monitoring the quality of machine learning models. Users can set up monitoring jobs to evaluate their model's performance at scheduled intervals, capture raw data, and visualize results, which helps in identifying and addressing issues in real time.

The "What is Evidently?" document outlines the platform's overarching purpose: to evaluate, test, and monitor AI-powered systems. It highlights features like the open-source library, the cloud platform for AI observability, and resources for monitoring ML models, assisting users in understanding how to get started.

In summary, to monitor your machine learning models using Evidently, you should:

1. Set up batch monitoring jobs to regularly evaluate your models' performance.
2. Utilize scheduled evaluations to trace and analyze dat

In [20]:
messages = result.all_messages()

In [21]:
result.usage()

RunUsage(input_tokens=3900, cache_read_tokens=1024, output_tokens=446, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=3, tool_calls=3)

In [22]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: how do I use evidently to monitor my machine learning models?

response
TOOL CALL: search {"query":"monitor machine learning models"}

request
TOOL RETURN: search

response
The search query "monitor machine learning models" is appropriate because it directly addresses the user's need to understand how to monitor machine learning models using the platform's features. The results of this search likely include documents that outline the various functionalities, best practices, and specific methods for effective monitoring.

Next, I will analyze the results to determine the most relevant documents for the user's question. The document titled "Overview" appears promising because it likely discusses methods for setting up monitoring. Therefore, I will conduct a get_file call to retrieve this document for more detailed insights. Additionally, I will also check the document titled "What is Evidently?" as it may provide foundational understanding of the monitoring capabilities. 



In [25]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [26]:
from pydantic_ai import RunUsage

messages = []
usage = RunUsage()

while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())


request
USER: create dashbord

response
TOOL CALL: search {"query":"create dashboard"}

request
TOOL RETURN: search

response
1) The search query "create dashboard" is appropriate as it directly relates to the user's request to create a dashboard. This query aims to locate documentation that provides guidelines or steps for the creation of dashboards, whether through a UI or API.

2) Analyzing the results from the previous search, the most relevant documents include "Add dashboard panels (UI)" and "Add dashboard panels (API)", as they specifically address how to design and create dashboards in different contexts. I will perform two additional searches with the queries "dashboard overview" and "dashboard management" to gather more comprehensive information. Additionally, I will retrieve the file "docs/platform/dashboard_overview.mdx" as it likely contains foundational information about what a dashboard is and how it functions.

Here are the specific actions I will take for this iteratio

In [27]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)


In [28]:
callback = NamedCallback(search_agent)

In [29]:
result = await search_agent.run(
    "how do I build dashboads with evidently?",
    event_stream_handler=callback
)

TOOL CALL (search): search({"query":"build dashboards"})
TOOL CALL (search): search({"query": "dashboard creation"})
TOOL CALL (search): search({"query": "dashboard examples"})
TOOL CALL (search): get_file({"filename": "docs/platform/dashboard_add_panels.mdx"})
TOOL CALL (search): get_file({"filename":"docs/platform/dashboard_add_panels.mdx"})
